# Mini RL Projesi: Yılan Oyununu Sıfırdan Kurup Bir Ajana Öğrettim

[haticeozbolat.com](https://haticeozbolat.com) yazısının not defteri. Yazının tamamı: https://haticeozbolat.com/?p=1067

Bu not defterinde:
1. Yılan oyununu Gymnasium ortamı olarak yazıyoruz.
2. Rastgele ve kural tabanlı iki kıyas ajanı kuruyoruz.
3. Tablo tabanlı Q-learning ile bir ajan eğitiyoruz (11 bilgilik gözlem).
4. Gözleme "sıkışır mıyım?" bilgisini ekleyip tekrar eğitiyoruz (14 bilgi).
5. Sonuçları karşılaştırıyor, öğrenme eğrisini çiziyor ve ajanın bir oyununu GIF olarak kaydediyoruz.

Sadece `numpy`, `gymnasium`, `matplotlib` ve `Pillow` kullanılıyor. Hücreleri sırayla **Shift + Enter** ile çalıştır.

In [ ]:
!pip -q install gymnasium

## 1. Ortam: YilanOyunu

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

# Yönler: 0 sağ, 1 aşağı, 2 sol, 3 yukarı (saat yönünde sıralı)
YONLER = [(1, 0), (0, 1), (-1, 0), (0, -1)]


class YilanOyunu(gym.Env):
    """Klasik yılan oyunu, Gymnasium ortamı olarak.

    Eylemler (yılanın kendi yönüne göre): 0 düz git, 1 sağa dön, 2 sola dön
    Gözlem: 11 evet/hayır bilgisi
      [önü tehlikeli, sağı tehlikeli, solu tehlikeli,
       sağa gidiyor, aşağı gidiyor, sola gidiyor, yukarı gidiyor,
       yem solda, yem sağda, yem yukarıda, yem aşağıda]
    Ödül: yem +10, ölüm -10; yeme yaklaşınca +yakinlik, uzaklaşınca -yakinlik
    """

    metadata = {"render_modes": ["ansi"]}

    def __init__(self, boyut=10, yakinlik_odulu=0.1, alan_bilgisi=False, render_mode=None):
        super().__init__()
        self.boyut = boyut
        self.yakinlik_odulu = yakinlik_odulu
        self.alan_bilgisi = alan_bilgisi          # True: 3 "sıkışır mıyım?" bilgisi daha ekle
        self.render_mode = render_mode
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.MultiBinary(14 if alan_bilgisi else 11)

    # --- yardımcılar ---
    def _yem_koy(self):
        bos = [(x, y) for x in range(self.boyut) for y in range(self.boyut) if (x, y) not in self.govde_kume]
        self.yem = bos[self.np_random.integers(len(bos))]

    def _carpar_mi(self, nokta):
        x, y = nokta
        duvar = not (0 <= x < self.boyut and 0 <= y < self.boyut)
        kendine = nokta in self.govde_kume and nokta != self.govde[-1]  # kuyruk o adımda çekilir
        return duvar or kendine

    def _ileri(self, yon):
        dx, dy = YONLER[yon]
        x, y = self.govde[0]
        return (x + dx, y + dy)

    def _gozlem(self):
        bas = self.govde[0]
        sag, sol = (self.yon + 1) % 4, (self.yon - 1) % 4
        g = [
            self._carpar_mi(self._ileri(self.yon)),
            self._carpar_mi(self._ileri(sag)),
            self._carpar_mi(self._ileri(sol)),
            self.yon == 0, self.yon == 1, self.yon == 2, self.yon == 3,
            self.yem[0] < bas[0], self.yem[0] > bas[0],
            self.yem[1] < bas[1], self.yem[1] > bas[1],
        ]
        if self.alan_bilgisi:
            for yon in (self.yon, sag, sol):
                hedef = self._ileri(yon)
                g.append(not self._carpar_mi(hedef) and self._serbest_alan(hedef) < len(self.govde))
        return np.array(g, dtype=np.int8)

    def _serbest_alan(self, baslangic):
        """O kareden başlayıp kaç boş kareye ulaşılabilir? (taşma doldurma / flood fill)"""
        engel = set(self.govde[:-1])            # kuyruk bir sonraki adımda çekileceği için hariç
        gorulen, yigin = {baslangic}, [baslangic]
        while yigin:
            x, y = yigin.pop()
            for dx, dy in YONLER:
                n = (x + dx, y + dy)
                if n not in gorulen and n not in engel and 0 <= n[0] < self.boyut and 0 <= n[1] < self.boyut:
                    gorulen.add(n); yigin.append(n)
        return len(gorulen)

    def _uzaklik(self):
        return abs(self.govde[0][0] - self.yem[0]) + abs(self.govde[0][1] - self.yem[1])

    # --- Gymnasium API ---
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        o = self.boyut // 2
        self.govde = [(o, o), (o - 1, o), (o - 2, o)]   # baş en başta
        self.govde_kume = set(self.govde)
        self.yon = 0
        self.skor = 0
        self.ac_adim = 0                                  # son yemden beri geçen adım
        self._yem_koy()
        return self._gozlem(), {"skor": 0}

    def step(self, eylem):
        if eylem == 1:
            self.yon = (self.yon + 1) % 4
        elif eylem == 2:
            self.yon = (self.yon - 1) % 4
        once = self._uzaklik()
        yeni_bas = self._ileri(self.yon)
        self.ac_adim += 1

        if self._carpar_mi(yeni_bas):
            return self._gozlem(), -10.0, True, False, {"skor": self.skor, "sebep": "carpti"}

        self.govde.insert(0, yeni_bas)
        self.govde_kume.add(yeni_bas)
        if yeni_bas == self.yem:
            self.skor += 1
            self.ac_adim = 0
            odul = 10.0
            if len(self.govde) == self.boyut ** 2:
                return self._gozlem(), odul, True, False, {"skor": self.skor, "sebep": "kazandi"}
            self._yem_koy()
        else:
            kuyruk = self.govde.pop()
            self.govde_kume.discard(kuyruk)
            odul = self.yakinlik_odulu if self._uzaklik() < once else -self.yakinlik_odulu

        # Sonsuz dönmeyi engelle: uzun süre yem yemezse bölümü kes
        kesildi = self.ac_adim > 100 * len(self.govde)
        return self._gozlem(), odul, False, kesildi, {"skor": self.skor}

    def render(self):
        satirlar = []
        for y in range(self.boyut):
            s = ""
            for x in range(self.boyut):
                if (x, y) == self.govde[0]: s += "O"
                elif (x, y) in self.govde_kume: s += "o"
                elif (x, y) == self.yem: s += "*"
                else: s += "."
            satirlar.append(s)
        return "\n".join(satirlar)


def durum_no(gozlem):
    """Bitlerden oluşan gözlemi tek bir tamsayıya çevirir (Q tablosunun satırı)."""
    return int("".join(map(str, gozlem)), 2)


In [ ]:
from gymnasium.utils.env_checker import check_env
check_env(YilanOyunu(), skip_render_check=True)
check_env(YilanOyunu(alan_bilgisi=True), skip_render_check=True)
print("Ortam Gymnasium arayüzüne uyuyor.")

env = YilanOyunu()
gozlem, _ = env.reset(seed=0)
print(env.render())
print("Gözlem:", gozlem)

## 2. Ajanlar ve Q-learning

In [ ]:
import numpy as np

import numpy as np


def rastgele_ajan(gozlem, rng):
    return int(rng.integers(3))


def kural_ajani(gozlem, rng):
    """Elle yazılmış kural: tehlikeli olmayan yönler arasından yeme yaklaştıranı seç."""
    on, sag, sol = gozlem[0], gozlem[1], gozlem[2]
    yon = int(np.argmax(gozlem[3:7]))                   # 0 sağ, 1 aşağı, 2 sol, 3 yukarı
    yem_sol, yem_sag, yem_yukari, yem_asagi = gozlem[7:11]
    istenen = []                                        # yeme götüren mutlak yönler
    if yem_sag: istenen.append(0)
    if yem_asagi: istenen.append(1)
    if yem_sol: istenen.append(2)
    if yem_yukari: istenen.append(3)
    adaylar = []
    for eylem, tehlike, yeni_yon in [(0, on, yon), (1, sag, (yon + 1) % 4), (2, sol, (yon - 1) % 4)]:
        if not tehlike:
            adaylar.append((yeni_yon in istenen, eylem))
    if not adaylar:
        return 0
    adaylar.sort(reverse=True)
    return adaylar[0][1]


def oyna(env, politika, bolum=200, tohum=10_000):
    rng = np.random.default_rng(tohum)
    skorlar = []
    for b in range(bolum):
        g, _ = env.reset(seed=tohum + b)
        bitti = kesildi = False
        while not (bitti or kesildi):
            g, _, bitti, kesildi, bilgi = env.step(politika(g, rng))
        skorlar.append(bilgi["skor"])
    return np.array(skorlar)


def q_ogren(env, bolum=3000, alfa=0.1, gama=0.9, eps_bas=1.0, eps_son=0.01, eps_azalma=0.995, tohum=0):
    rng = np.random.default_rng(tohum)
    Q = np.zeros((2 ** env.observation_space.n, 3))
    eps = eps_bas
    gecmis = []
    for b in range(bolum):
        g, _ = env.reset(seed=tohum + b)
        s = durum_no(g)
        bitti = kesildi = False
        while not (bitti or kesildi):
            if rng.random() < eps:
                a = int(rng.integers(3))
            else:
                a = int(np.argmax(Q[s]))
            g, r, bitti, kesildi, bilgi = env.step(a)
            s2 = durum_no(g)
            hedef = r if bitti else r + gama * np.max(Q[s2])     # Bellman güncellemesi
            Q[s, a] += alfa * (hedef - Q[s, a])
            s = s2
        eps = max(eps_son, eps * eps_azalma)
        gecmis.append(bilgi["skor"])
    return Q, np.array(gecmis)


def q_politika(Q):
    return lambda g, rng: int(np.argmax(Q[durum_no(g)]))


## 3. Kıyas çizgileri

In [ ]:
env11 = YilanOyunu(boyut=10)
sonuc = {}
sonuc["Rastgele"] = oyna(env11, rastgele_ajan, bolum=300)
sonuc["Kural tabanlı"] = oyna(env11, kural_ajani, bolum=300)
for ad, s in sonuc.items():
    print(f"{ad:<15} ortalama {s.mean():5.2f}  medyan {np.median(s):4.0f}  en iyi {s.max()}")

## 4. Q-learning, 11 bilgilik gözlem (yaklaşık 10 saniye)

In [ ]:
import time
t = time.time()
Q11, gecmis11 = q_ogren(env11, bolum=5000, tohum=0)
print(f"Eğitim {time.time() - t:.0f} saniye sürdü")
sonuc["Q-ajan (11 bilgi)"] = oyna(env11, q_politika(Q11), bolum=300)
print("Ziyaret edilen durum sayısı:", int((Q11 != 0).any(axis=1).sum()), "/", Q11.shape[0])

## 5. Gözleme "sıkışır mıyım?" bilgisini eklemek (yaklaşık 3 dakika)

`alan_bilgisi=True` ile gözlem 14 bilgiye çıkar: düz, sağa ya da sola gidersem ulaşabileceğim boş alan gövdemden küçük mü?

In [ ]:
env14 = YilanOyunu(boyut=10, alan_bilgisi=True)
t = time.time()
Q14, gecmis14 = q_ogren(env14, bolum=5000, tohum=0)
print(f"Eğitim {time.time() - t:.0f} saniye sürdü")
sonuc["Q-ajan (14 bilgi)"] = oyna(env14, q_politika(Q14), bolum=300)

## 6. Sonuçlar

In [ ]:
print(f"{'Ajan':<20}{'Ortalama':>10}{'Medyan':>8}{'En iyi':>8}")
for ad, s in sonuc.items():
    print(f"{ad:<20}{s.mean():>10.2f}{np.median(s):>8.0f}{s.max():>8}")

In [ ]:
import matplotlib.pyplot as plt
MURDUM, SEFTALI, BOGURTLEN = "#2B1036", "#F6B17A", "#C2185B"

def kayan_ortalama(x, n=250):
    return [x[i:i + n].mean() for i in range(0, len(x), n)]

eksen_x = np.arange(250, 5001, 250)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(eksen_x, kayan_ortalama(gecmis11), color="#B8B2BC", lw=2, label="Q-ajan (11 bilgi)")
ax.plot(eksen_x, kayan_ortalama(gecmis14), color=BOGURTLEN, lw=2.5, label="Q-ajan (14 bilgi)")
ax.axhline(sonuc["Kural tabanlı"].mean(), color=MURDUM, ls="--", lw=1, label="Kural tabanlı (test)")
ax.set_xlabel("Eğitim oyunu"); ax.set_ylabel("Ortalama yem (250 oyunluk)")
ax.set_title("Eğitim sırasında öğrenme eğrisi (keşif dahil)", color=MURDUM)
ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False)
plt.show()

## 7. Ajanın bir oyununu GIF olarak kaydet

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Image as GosterImage

def gif_kaydet(env, Q, tohum, dosya="yilan_ajan.gif", H=40, UST=50):
    try:
        font = ImageFont.load_default(size=22)
    except TypeError:                       # eski Pillow sürümleri
        font = ImageFont.load_default()
    g, _ = env.reset(seed=tohum); kareler = []; bitti = kesildi = False
    while not (bitti or kesildi):
        n = env.boyut
        im = Image.new("RGB", (H * n, H * n + UST), MURDUM); d = ImageDraw.Draw(im)
        for x in range(n):
            for y in range(n):
                d.rectangle([x*H+2, UST+y*H+2, x*H+H-2, UST+y*H+H-2], fill="#3B1A4A")
        fx, fy = env.yem
        d.ellipse([fx*H+7, UST+fy*H+7, fx*H+H-7, UST+fy*H+H-7], fill=BOGURTLEN)
        for i, (x, y) in enumerate(env.govde):
            d.rectangle([x*H+3, UST+y*H+3, x*H+H-3, UST+y*H+H-3], fill=SEFTALI if i else "white")
        d.text((12, 14), f"Skor: {env.skor}", font=font, fill=SEFTALI)
        kareler.append(im)
        g, _, bitti, kesildi, _ = env.step(int(np.argmax(Q[durum_no(g)])))
    kareler += [kareler[-1]] * 15
    kareler[0].save(dosya, save_all=True, append_images=kareler[1:], duration=70, loop=0)
    return env.skor

# 20 oyun arasından en uzununu seçip kaydet
en_iyi_tohum = max(range(20000, 20020), key=lambda t: oyna(env14, q_politika(Q14), bolum=1, tohum=t)[0])
skor = gif_kaydet(env14, Q14, tohum=en_iyi_tohum)
print("Kaydedilen oyunun skoru:", skor)
GosterImage(filename="yilan_ajan.gif")

## Kendin dene

- `YilanOyunu(boyut=8)` ya da `boyut=12` ile tahtayı küçült/büyüt.
- `yakinlik_odulu=0.0` ile ödül şekillendirmeyi kapat ve sonucu karşılaştır.
- `q_ogren` içinde `gama` ve `alfa` değerlerini değiştir.
- Kural tabanlı ajana da "tuzak olan yöne gitme" kuralını ekle: 14 bilgili Q-ajanla arasındaki fark kapanıyor mu?